SCDs refer to data in dimension tables that changes slowly over time and not
at a regular cadence.

*Designing SCD1*

In SCD type 1, the values are overwritten and no history is maintained, so
once the data is updated, there is no way to find out what the previous value
was. The new queries will always return the most recent value. Here is an
example of an SCD1 table:


#### Implementation: SCD-1
* Read the Data

In [0]:
# list of employee data 
data = [["1001", "gaurav", "hyderabad","42000"], 
        ["1002", "vijay", "hyderabad","45565"], 
        ["1003", "akanksha","hyderabad", "52000"], 
        ["1004", "niharika", "hyderabad","35000"]] 
# specify column names 
columns = ['id','name','location','salry'] 
# creating a dataframe from the lists of data 
df_full = spark.createDataFrame(data, columns) 
# list of employee data 
data = [ ["1003", "akanksha","delhi", "65000"], 
        ["1004", "niharika", "bihar","10000"],
        ["1005", "murali","vijaywada", "80000"],
        ["1002", "vijay", "hyderabad","45565"]
       ] 
# specify column names 
columns = ['id','name','location','salry'] 
# creating a dataframe from the lists of data 
df_daily_update = spark.createDataFrame(data, columns)

In [0]:
print("Full data...")
df_full.show()
print("daily data...")
df_daily_update.show()

#### Insert & Update Operation

In [0]:
from pyspark.sql.functions import coalesce
res=df_full.join(df_daily_update,"id","full_outer").\
select(coalesce(df_full.id,df_daily_update.id).alias("ID"),\
       coalesce(df_daily_update.name,df_full.name).alias("Name"),\
       coalesce(df_daily_update.location,df_full.location).alias("Location"),\
        coalesce(df_daily_update.salry,df_full.salry).alias("Salary")
       )
res.show()

### SCD Type-2 

Type 2 dimensions are always created as a new record. If a detail in the data changes, a new row will be added to 
the table with a new primary key. However, the natural key would remain the same in order to map a record 
change to one another. Type 2 dimensions are the most common approach to tracking historical records

#### SCD 2 Implementation

* Read the data into dataframes

In [0]:
import pyspark 
from pyspark.sql import SparkSession 
spark = SparkSession.builder.appName('sparkdf').getOrCreate() 
# list of employee data 
data = [["1001", "gaurav", "hyderabad","42000"], 
        ["1002", "vijay", "hyderabad","45565"], 
        ["1003", "akanksha","hyderabad", "52000"], 
        ["1004", "niharika", "hyderabad","35000"]] 
# specify column names 
columns = ['id','name','location','salry'] 
# creating a dataframe from the lists of data 
df_full = spark.createDataFrame(data, columns) 

In [0]:
display(df_full)

In [0]:
# list of employee data 
data = [ ["1003", "akanksha","delhi", "65000"], 
        ["1004", "niharika", "bihar",None],
        ["1005", "murali","vijaywada", "80000"],
        ["1002", "vijay", "hyderabad","45565"]
       ] 
# specify column names 
columns = ['id','name','location','salry'] 
# creating a dataframe from the lists of data 
df_daily = spark.createDataFrame(data, columns)

In [0]:
display(df_daily)

#### Adding Additional Columns in both DataFrames

In [0]:
from pyspark.sql.functions import *
df_full=df_full.withColumn("Active_Flag",lit("Y")).withColumn("From_date",\
to_date(current_date()))\
.withColumn("To_date",lit("Null"))
df_full.show()

In [0]:
df_daily=df_daily.withColumn("Active_Flage",lit("Y"))\
.withColumn("From_date",to_date(current_date()))\
.withColumn("To_date",lit("Null"))
display(df_daily)

#### Create Dataframe by using Updating the Active Flag If any changes in dataframes using 
Hash

In [0]:
update_ds=df_full.join(df_daily,((df_full.id==df_daily.id) & (df_full.Active_Flag =='Y')), 
"inner")\
    .filter(hash(df_full.name,df_full.location,df_full.salry ) !=
            hash(df_daily.name,df_daily.location,df_daily.salry  ) )\
                .select(df_full.id,
                        df_full.name,
                        df_full.location,
                        df_full.salry,
                        lit("N").alias("Active_Flag"),
                        df_full.From_date,
                        lit(to_date(current_date())).alias("To_Date"))
update_ds.show()

#### Create a Data Frame with No changes data using the update_ds dataframe

In [0]:
no_change=df_full.join(update_ds,((df_full.id==update_ds.id) & (df_full.Active_Flag =='Y')),
"left_anti")
no_change.show()

#### Create data frame using no_change Dataframe and the dataframe consists of the new 
records that need to insert

In [0]:
insert_ds = df_daily.join(no_change,"id","left_anti")
insert_ds.show()

#### Finally We have to concatenate update_ds , Insert_ds, no_change dataframe by using the 
union function. 

In [0]:
update_ds_fixed = update_ds.withColumn("To_Date", update_ds.To_Date.cast("string"))
df_final=update_ds_fixed.union(insert_ds).union(no_change)
df_final.show()